<a href="https://colab.research.google.com/github/ivorywang-1005/intro-to-urban-data/blob/main/Scale/S09_Scale_Calling_an_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **09 Scale**  Calling an API to get external data

```
    Class       ARCH 6131 Urban Design Methods / Skills / Tools 1
                Tata Innovation Center
                Fall 2025
                Monday 9:30 AM to 12 PM

    Instructor  Brian Ho
                brian@brian-ho.io
```

This notebook is a quick demonstration of how you can call an external Application Programming Interface (API) to fetch data from providers and live sources, rather than just downloading existing files.

If you'd like to incoporate an API into your coursework, let me know! There are some nuanced to setting them up that we can review together.

## About APIs
APIs are made available by a variety of public and private sources, either for free or as a paid offering. One example that we'll use in this notebook are APIs from HERE (https://developer.here.com/) a company that provides geospatial information. Other APIs you might explore include:
- Google Maps Platform (https://mapsplatform.google.com/)
- Mapbox (https://www.mapbox.com/)
- Mapillary (https://www.mapillary.com/developer)

... and more! The internet is truly your oyster — being able to call APIs enables you to access a number of "developer" tools. Of course, that comes with the caveat that things get a little tricker to work with!


## Setting up with HERE
To set up the HERE API, go to https://developer.here.com/ and create your free account. After you do that, you'll need to:
1. Create a project in the [Project Manger](https://platform.here.com/management/projects), giving it access to relevant services, e.g. "HERE Search - One Box Search".
2. Create an app in the [Access Manager](https://platform.here.com/admin/apps), giving it access to the project you created.

You'll use the API key for your app to access data. So long as you stay under 1,000 calls a day, your access wil be free.

## Call an API from your notebook

Calling the API is quite simple using the built-in `requests` library. This comes built-in to Colab Notebooks!

In [ ]:
# We'll use requests to call the API and time to manage delays
import requests
import time

Essentially, you can pass in the base URL (entrypoint and endpoint) as well as any parameters needed to complete your API request.

For each API, you will need to consult the "developer documentation" and "API reference" to fully understand the API and work with it successfully. In our case, we'll use the [HERE Search - One Box Search Geocoding API](https://platform.here.com/services/details/hrn:here:service::olp-here:search-discover-7/api-ref) from HERE. In principal, each API call wil have 3 parts:

1. The entrypoint which defines the overall API provider
2. The endpoint which defines the specific service
3. Parameters which define the API call's specific requests

In our example, we can us the `discover` endpoint on the `discover.search.hereapi.com/v1` entrpoint to find potential place of interest near a given location and matching a query string with the `at` and `q` paramters.

In [ ]:
from google.colab import userdata
HERE_API_KEY = userdata.get("HERE_API_KEY")


In [ ]:
# Discover
def search_for_places(latitude, longitude, query):
    """A helper function that calls the HERE Discover API."""

    print(f"Searching for {query} at {latitude}, {longitude} ...")

    # API often limit rate of access — this slows things down
    time.sleep(0.1)

    # Use requests to set query parameters per API documentation
    params = {
        "at": f"{latitude},{longitude}",
        # Don't forget to enter your API key here!
        "apiKey": HERE_API_KEY,
        "q": query,
    }

    # Make the request and return response
    return requests.get(
        "https://discover.search.hereapi.com/v1/discover", params=params
    )

That's it! We can test this out with an example query.

In [ ]:
# Try out the API call for 26 Broadway in Manhattan, NYC
api_result = search_for_places(40.883740213197676, -74.77109109191505, "school")
api_result_data = api_result.json()

print(f"Found {len(api_result_data['items'])} places!")

The full response is a lot of JSON! But we can parse this like any Python data structure.

In [ ]:
api_result_data

In [ ]:
for place in api_result_data["items"]:
    print(place["address"]["label"])

In [ ]:
#Autocomplete
import requests
def autocomplete(query, country=None):
  "Call the HERE autocomplete API and return the JSON response"
  params = {
      "q": query,
      "apiKey":HERE_API_KEY,
  }
  response = requests.get("https://autocomplete.search.hereapi.com/v1/autocomplete", params=params)
  print(response.status_code)
  return response.json()


In [ ]:
api_result = autocomplete("Jersey Ci")
api_result

## Integrating APIs into a dataframe

Now you can pretty easily use this same logic in a `apply` function!

In [ ]:
# Connect to Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import geopandas as gpd

In [ ]:
# Get 10 random lots from an existing dataset
ten_vacant_lots = gpd.read_file(
    "/content/drive/MyDrive/ARCH 6131/Example Data/brooklyn_city_vacant_lots.geojson"
).sample(10)
ten_vacant_lots.plot()

In [ ]:
def search_for_coffee(row):
    """Gets first place that matches query for coffee from HERE Discover API"""

    # Get the middle point of the lot
    pt = row.geometry.centroid
    latitude = pt.y
    longitude = pt.x

    # Keep under rate limits
    time.sleep(0.1)

    # Set up the request
    params = {
        "at": f"{latitude},{longitude}",
        # Don't forget to enter your API key here!
        "apiKey": HERE_API_KEY,
        "q": "coffee",
    }

    # Make the request and return the first address
    r = requests.get("https://discover.search.hereapi.com/v1/discover?", params=params)
    return r.json()["items"][0]["address"]["label"]

In [ ]:
# Apply the function! Cafeful — you'll only have 1,000 free requests with HERE's APIs
ten_vacant_lots["coffee"] = ten_vacant_lots.apply(search_for_coffee, axis=1)

In [ ]:
# That's it!
ten_vacant_lots[["geometry", "coffee"]]